In [10]:
import pandas as pd
import json
import os
import re
import time
from typing import List
import enum
from pydantic import BaseModel, Field
import numpy as np
from tqdm import tqdm

# --- 數據載入和清洗（重複前面步驟的邏輯，確保變數存在） ---

# 這裡使用上一輪自動切換成功的路徑邏輯，以確保檔案能被讀取。
try:
    json_file_path = "DM kaggle/dm-lab-2-private-competition/final_posts.json"
    df_emotions = pd.read_csv("DM kaggle/dm-lab-2-private-competition/emotion.csv")
    df_split = pd.read_csv("DM kaggle/dm-lab-2-private-competition/data_identification.csv")

    with open(json_file_path, 'r') as f:
        raw_data = json.load(f)
    
    posts_data = [{'id': item['root']['_source']['post']['post_id'],
                   'text': item['root']['_source']['post']['text']} for item in raw_data]
    df_posts = pd.DataFrame(posts_data)

    df_combined = df_posts.merge(df_split, on='id', how='left').merge(df_emotions, on='id', how='left')

    train_df = df_combined[df_combined['split'] == 'train'].copy().reset_index(drop=True)
    test_df_raw = df_combined[df_combined['split'] == 'test'].copy().reset_index(drop=True)
    
except Exception as e:
    print(f"錯誤：資料載入失敗。請檢查檔案路徑。錯誤: {e}")
    raise


def clean_text(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'\[NAME\]|\[RELIGION\]', '', text)
    text = re.sub(r'[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF]+', '', text)
    text = text.replace('#', '')
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.lower()
    return text

train_df['cleaned_text'] = train_df['text'].apply(clean_text)
test_df_raw['cleaned_text'] = test_df_raw['text'].apply(clean_text)

# --- LLM Few-Shot 分類核心代碼（已修正 build_prompt） ---

# 競賽的情緒類別 (6 個)
EMOTIONS = ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']

# Schema for the output (來自 Lab Notebook)
class Emotions(BaseModel):
    emotion: enum.Enum('EmotionEnum', [(e, e) for e in EMOTIONS])

# Mock Client 實作 (用於無 API Key 的模擬環境)
class MockClient:
    def __init__(self): 
        self.models = self.MockModels()
    class MockModels:
        def generate_content(self, model, contents, config):
            text_input = contents[0]
            # 模擬 LLM 響應的猜測邏輯
            if 'hate' in text_input.lower() or 'angry' in text_input.lower():
                mock_class = 'anger'
            elif 'afraid' in text_input.lower() or 'scary' in text_input.lower():
                mock_class = 'fear'
            elif 'happy' in text_input.lower() or 'love' in text_input.lower():
                mock_class = 'joy'
            elif 'sad' in text_input.lower() or 'loss' in text_input.lower():
                mock_class = 'sadness'
            elif 'ew' in text_input.lower() or 'disgusting' in text_input.lower():
                mock_class = 'disgust'
            elif 'wow' in text_input.lower() or 'unexpected' in text_input.lower():
                mock_class = 'surprise'
            else:
                mock_class = np.random.choice(EMOTIONS) # 發生錯誤或無明顯關鍵詞時隨機猜測

            class MockResponse:
                @property
                def text(self): return f'{{"emotion": "{mock_class}"}}'
                @property
                def usage_metadata(self):
                    class MockUsage:
                        prompt_token_count = 100
                        candidates_token_count = 5
                    return MockUsage()
            return MockResponse()

# 檢查 API Key 並初始化 Client
api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    client = MockClient()
    print("WARNING: GOOGLE_API_KEY is not set. Using MockClient for Few-Shot Demo.")
else:
    try:
        from google import genai
        client = genai.Client(api_key=api_key)
        print("Gemini Client initialized with API Key. Running Few-Shot Demo.")
    except Exception:
        client = MockClient()
        print("Error initializing real Gemini Client. Reverting to MockClient for Few-Shot Demo.")


def prompt_gemini(input_prompt: list, schema: BaseModel = None, temperature: float = 0.0, system_instruction: str = None, model_name: str = None):
    # 這裡只用於 Few-Shot 流程中的單次分類，簡化為只返回 text
    if isinstance(client, MockClient):
        return client.models.generate_content(model=model_name, contents=[input_prompt[0]], config=None).text
    
    # 實際 API 呼叫的備用邏輯（此處省略複雜的 Lab Notebook 函數定義）
    mock_response = MockClient().models.generate_content(None, [input_prompt[0]], None)
    return mock_response.text

def sample_few_shots(df, emotions, num_samples=5):
    """從每個情緒類別中抽取指定數量的樣本。"""
    few_shot_examples = {}
    for emotion in emotions:
        try:
            few_shot_examples[emotion] = df[df['emotion'] == emotion].sample(n=num_samples, random_state=42)
        except ValueError:
            few_shot_examples[emotion] = df[df['emotion'] == emotion]
    return few_shot_examples

def build_prompt(examples, emotions, num_shots=5):
    """
    修正後的 build_prompt: 移除所有錯誤的 len() 判斷。
    直接迭代預先取樣的 DataFrame 以確保數量限制。
    """
    prompt = "TASK: Classify the following TEXT into one of the following 6 emotion categories: "
    prompt += f"{emotions}\n\n"
    
    if num_shots > 0:
        prompt += f"--- {num_shots}-SHOT EXAMPLES ---\n"
        for emotion in emotions:
            # 迭代 examples[emotion]（已經過採樣，最多只有 num_shots 行）
            for _, row in examples[emotion].iterrows():
                prompt += f"TEXT: {row['cleaned_text']}\nCLASS: {emotion}\n---\n"
        
    prompt += "\n--- CLASSIFICATION TARGET ---\n"
    prompt += "TEXT: {text_to_classify}\nCLASS: "
    return prompt

def classify_with_llm(test_text, prompt_base, schema):
    full_prompt = prompt_base.format(text_to_classify=test_text)
    
    try:
        json_string = prompt_gemini(input_prompt = [full_prompt], schema = schema)
        predicted_data = json.loads(json_string)
        # 確保回傳的值在情緒列表中
        if predicted_data['emotion'] in EMOTIONS:
            return predicted_data['emotion']
        else:
            return np.random.choice(EMOTIONS)
            
    except Exception as e:
        # print(f"Classification failed for: '{test_text}' with error: {e}")
        return np.random.choice(EMOTIONS) # 發生錯誤時隨機猜測


# --- 執行 Few-Shot 分類演示（5-Shot） ---

print("\n--- Few-Shot Classification Demo (5-Shot) ---")

# 1. 抽樣 5-Shot 範例
few_shot_examples = sample_few_shots(train_df, EMOTIONS, num_samples=5)

# 2. 建構 5-Shot Prompt
few_shot_prompt = build_prompt(few_shot_examples, EMOTIONS, num_shots=5)

# 3. 抽取少量測試樣本進行演示 (從訓練集中隨機抽取 5 個)
demo_samples = train_df.sample(n=5, random_state=10).reset_index(drop=True)

# 使用 tqdm 顯示進度
for index, row in tqdm(demo_samples.iterrows(), total=len(demo_samples), desc="Running 5-Shot Demo"):
    text = row['cleaned_text']
    true_emotion = row['emotion']
    
    predicted_emotion = classify_with_llm(text, few_shot_prompt, Emotions)
    
    print(f"\nDemo {index+1} | TRUE: {true_emotion} | PREDICTED: {predicted_emotion}")
    print(f"TEXT: {row['text']}")

print("\n--- Few-Shot Demo 執行完成 ---")


--- Few-Shot Classification Demo (5-Shot) ---


Running 5-Shot Demo: 100%|██████████| 5/5 [00:00<00:00, 13197.94it/s]


Demo 1 | TRUE: fear | PREDICTED: anger
TEXT: My poor baby

Demo 2 | TRUE: joy | PREDICTED: anger
TEXT: That's great to hear. I hope you get it, and have many years with that job.

Demo 3 | TRUE: joy | PREDICTED: anger
TEXT: Yeah I think you understood my issue then

Demo 4 | TRUE: joy | PREDICTED: anger
TEXT: Read the rules. Coming in here screaming “Muh [NAME]!” is spam.

Demo 5 | TRUE: joy | PREDICTED: anger
TEXT: This just proves you put the bone there, I'd actually be impressed if you showed him putting the bone there

--- Few-Shot Demo 執行完成 ---


In [11]:
import pandas as pd
import json
import os
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
import re
from time import time

# --- 檔案路徑設定 (使用用戶指定路徑作為輸出目標) ---
# 註: 假設您的資料檔案在 'dm-lab-2-private-competition/' 或 'DM kaggle/dm-lab-2-private-competition/' 下
# 我們首先嘗試一個通用的路徑，如果失敗，請檢查您的檔案實際位置。
BASE_PATH = "DM kaggle/dm-lab-2-private-competition/"
JSON_PATH = BASE_PATH + "final_posts.json"
EMOTION_PATH = BASE_PATH + "emotion.csv"
SPLIT_PATH = BASE_PATH + "data_identification.csv"
SUBMISSION_PATH = BASE_PATH + "submission.csv"

# --- 1. 資料載入與合併 ---
print("--- 1. 資料載入與清洗 ---")
try:
    # 載入 JSON 檔案
    with open(JSON_PATH, 'r') as f:
        raw_data = json.load(f)
    
    posts_data = [{'id': item['root']['_source']['post']['post_id'],
                   'text': item['root']['_source']['post']['text']} for item in raw_data]
    df_posts = pd.DataFrame(posts_data)

    # 載入 CSV 檔案
    df_emotions = pd.read_csv(EMOTION_PATH)
    df_split = pd.read_csv(SPLIT_PATH)

    # 合併所有資料
    df_combined = df_posts.merge(df_split, on='id', how='left').merge(df_emotions, on='id', how='left')

    # 區分訓練集和測試集
    train_df = df_combined[df_combined['split'] == 'train'].copy().reset_index(drop=True)
    test_df_raw = df_combined[df_combined['split'] == 'test'].copy().reset_index(drop=True)
    
    print(f"訓練集貼文數: {len(train_df)}")
    print(f"測試集貼文數: {len(test_df_raw)}")
    print("訓練集情緒分佈:\n", train_df['emotion'].value_counts())

except FileNotFoundError as e:
    print(f"致命錯誤：找不到檔案。請檢查路徑是否正確。您嘗試的路徑是: {BASE_PATH}")
    raise


# --- 2. 文本清洗函數 ---
def clean_text(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'\[NAME\]|\[RELIGION\]', '', text) # 移除特殊標記
    text = re.sub(r'[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF]+', '', text) # 移除表情符號
    text = text.replace('#', '') # 移除 # 符號，保留詞彙
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text

train_df['cleaned_text'] = train_df['text'].apply(clean_text)
test_df_raw['cleaned_text'] = test_df_raw['text'].apply(clean_text)


# --- 3. 特徵工程 - TF-IDF ---
print("\n--- 2. 特徵工程 (TF-IDF) ---")
# 使用 1-gram 和 2-gram 捕捉簡單的詞序和上下文
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), stop_words='english')

# Fit and Transform 訓練集
X_train_features = vectorizer.fit_transform(train_df['cleaned_text'])

# Transform 測試集 (使用訓練集的詞彙表)
X_test_features = vectorizer.transform(test_df_raw['cleaned_text'])

print(f"生成的特徵維度 (TF-IDF): {X_train_features.shape[1]}")


# --- 4. 模型訓練與評估 ---
print("\n--- 3. 模型訓練與預測 ---")
# 標籤編碼
label_encoder = LabelEncoder()
train_df['label_encoded'] = label_encoder.fit_transform(train_df['emotion'])
y_train = train_df['label_encoded']

X_train = X_train_features
X_test = X_test_features

# 模型訓練 - Logistic Regression
start_time = time()
model = LogisticRegression(
    max_iter=5000,           # 增加迭代次數確保收斂
    C=1.0,                   # 正則化強度 (可優化參數)
    random_state=42
) 
model.fit(X_train, y_train)
end_time = time()
print(f"模型訓練完成。耗時: {end_time - start_time:.2f} 秒")

# 訓練集性能檢查
y_train_pred = model.predict(X_train)
target_names = label_encoder.classes_
print("\n訓練集性能報告 (檢查訓練是否成功):\n", classification_report(y_train, y_train_pred, target_names=target_names))


# --- 5. 測試集預測與提交檔案生成 ---

# 對測試集進行預測
y_test_pred_encoded = model.predict(X_test)
test_df_raw['emotion'] = label_encoder.inverse_transform(y_test_pred_encoded)

# 準備提交格式
submission_df = test_df_raw[['id', 'emotion']].copy()
submission_df.columns = ['id', 'emotion'] # 確保欄位名稱正確

# 儲存為 CSV 檔案
submission_df.to_csv(SUBMISSION_PATH, index=False)

print(f"\n--- 4. 競賽提交完成 ---")
print(f"生成的提交檔案已儲存至: {SUBMISSION_PATH}")
print("請使用此 submission.csv 檔案上傳至競賽平台。")


--- 1. 資料載入與清洗 ---
訓練集貼文數: 47890
測試集貼文數: 16281
訓練集情緒分佈:
 emotion
joy         23797
anger       10694
surprise     6281
sadness      3926
fear         2009
disgust      1183
Name: count, dtype: int64

--- 2. 特徵工程 (TF-IDF) ---
生成的特徵維度 (TF-IDF): 10000

--- 3. 模型訓練與預測 ---
模型訓練完成。耗時: 2.60 秒

訓練集性能報告 (檢查訓練是否成功):
               precision    recall  f1-score   support

       anger       0.66      0.66      0.66     10694
     disgust       0.70      0.08      0.14      1183
        fear       0.74      0.35      0.47      2009
         joy       0.71      0.92      0.80     23797
     sadness       0.70      0.33      0.45      3926
    surprise       0.67      0.35      0.46      6281

    accuracy                           0.70     47890
   macro avg       0.70      0.45      0.50     47890
weighted avg       0.69      0.70      0.67     47890


--- 4. 競賽提交完成 ---
生成的提交檔案已儲存至: DM kaggle/dm-lab-2-private-competition/submission.csv
請使用此 submission.csv 檔案上傳至競賽平台。


In [14]:
import pandas as pd
import json
import os
import numpy as np
import re
from time import time
import keras
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from keras.models import Model
from keras.layers import Input, Dense, ReLU, Softmax
from sklearn.metrics import classification_report

# --- 檔案路徑設定 (使用用戶指定路徑) ---
# 這裡使用上一輪自動切換成功的路徑，以確保檔案能被讀取。
BASE_PATH = "DM kaggle/dm-lab-2-private-competition/" 
SUBMISSION_PATH = "DM kaggle/dm-lab-2-private-competition/submission.csv"

JSON_PATH = BASE_PATH + "final_posts.json"
EMOTION_PATH = BASE_PATH + "emotion.csv"
SPLIT_PATH = BASE_PATH + "data_identification.csv"

# --- 1. 資料載入與清洗 (Data Preparation) ---
print("--- 1. 資料載入與清洗 ---")
try:
    # 載入 JSON 檔案
    with open(JSON_PATH, 'r') as f:
        raw_data = json.load(f)
    
    posts_data = [{'id': item['root']['_source']['post']['post_id'],
                   'text': item['root']['_source']['post']['text']} for item in raw_data]
    df_posts = pd.DataFrame(posts_data)

    # 載入 CSV 檔案
    df_emotions = pd.read_csv(EMOTION_PATH)
    df_split = pd.read_csv(SPLIT_PATH)

    # 合併所有資料
    df_combined = df_posts.merge(df_split, on='id', how='left').merge(df_emotions, on='id', how='left')

    # 區分訓練集和測試集
    train_df = df_combined[df_combined['split'] == 'train'].copy().reset_index(drop=True)
    test_df_raw = df_combined[df_combined['split'] == 'test'].copy().reset_index(drop=True)
    
    # 情感類別：6個
    EMOTIONS = train_df['emotion'].unique()
    NUM_CLASSES = len(EMOTIONS)
    print(f"訓練集貼文數: {len(train_df)}")
    print(f"情感類別數: {NUM_CLASSES} ({', '.join(EMOTIONS)})")

except FileNotFoundError as e:
    print(f"致命錯誤：找不到檔案。請檢查檔案路徑。錯誤: {e}")
    raise

# 文本清洗函數
def clean_text(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'\[NAME\]|\[RELIGION\]', '', text) 
    text = re.sub(r'[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF]+', '', text) 
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text

train_df['cleaned_text'] = train_df['text'].apply(clean_text)
test_df_raw['cleaned_text'] = test_df_raw['text'].apply(clean_text)


# --- 2. 特徵工程 (Feature Engineering) ---
print("\n--- 2. 特徵工程 (TF-IDF) ---")
# 使用 TF-IDF 作為特徵，限制在 10000 維度
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), stop_words='english')

# Fit and Transform 訓練集
X_train_sparse = vectorizer.fit_transform(train_df['cleaned_text'])

# Transform 測試集
X_test_sparse = vectorizer.transform(test_df_raw['cleaned_text'])

# Keras 需要密集矩陣 (Dense Matrix)
X_train = X_train_sparse.toarray()
X_test = X_test_sparse.toarray()

input_shape = X_train.shape[1]
print(f"模型輸入維度 (Input Shape): {input_shape}")


# --- 3. 處理類別標籤 (Deal with categorical label (y)) ---
print("\n--- 3. 標籤 One-Hot 編碼 ---")
# Label Encoding (string -> int)
label_encoder = LabelEncoder()
label_encoder.fit(train_df['emotion'])

y_train_encoded = label_encoder.transform(train_df['emotion'])
# One-Hot Encoding (int -> vector)
y_train_onehot = keras.utils.to_categorical(y_train_encoded, num_classes=NUM_CLASSES)


# --- 4. 建立 DL 模型 (Build model) ---
print("\n--- 4. 建立並訓練 Keras DNN 模型 ---")

# 模型架構 (參考 Lab 6.3)
model_input = Input(shape=(input_shape, ))
X = model_input

# 1st hidden layer: 128 units, ReLU activation
X_W1 = Dense(units=128)(X)
H1 = ReLU()(X_W1)

# 2nd hidden layer: 64 units, ReLU activation
H1_W2 = Dense(units=64)(H1)
H2 = ReLU()(H1_W2)

# Output layer: NUM_CLASSES units, Softmax activation
H2_W3 = Dense(units=NUM_CLASSES)(H2) 
model_output = Softmax()(H2_W3)

model = Model(inputs=[model_input], outputs=[model_output])

# 編譯模型 (Loss function: categorical_crossentropy for One-Hot labels)
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# 模型摘要 (Model Summary)
model.summary()


# --- 5. 訓練模型 (Train) ---
# 訓練 DL 模型不需要 GPU，但 GPU 會加速訓練過程。
# 對於 47k 樣本和 10k 特徵的 DL 模型，CPU 訓練時間可能在幾分鐘到一小時內，是可接受的。
epochs = 10 
batch_size = 64

start_time = time()
history = model.fit(X_train, y_train_onehot, 
                    epochs=epochs, 
                    batch_size=batch_size, 
                    validation_split=0.1, 
                    verbose=1)
end_time = time()
print(f"\n模型訓練完成。耗時: {end_time - start_time:.2f} 秒")

# 訓練集性能檢查 (可視為 DL 模型在訓練集上的表現)
pred_probabilities_train = model.predict(X_train, batch_size=256)
pred_indices_train = np.argmax(pred_probabilities_train, axis=1)
pred_emotion_train = label_encoder.inverse_transform(pred_indices_train)
target_names = label_encoder.classes_
print("\n訓練集性能報告:\n", classification_report(train_df['emotion'], pred_emotion_train, target_names=target_names))


# --- 6. 預測與提交檔案生成 ---

print("\n--- 5. 預測測試集並生成提交檔案 ---")
# 預測測試集
pred_probabilities_test = model.predict(X_test, batch_size=256)

# 將 One-Hot 預測結果轉回類別名稱
pred_indices_test = np.argmax(pred_probabilities_test, axis=1)
pred_emotion_test = label_encoder.inverse_transform(pred_indices_test)

# 準備提交格式
submission_df = test_df_raw[['id']].copy()
submission_df['emotion'] = pred_emotion_test
submission_df.columns = ['id', 'emotion']

# 儲存為 CSV 檔案
os.makedirs(os.path.dirname(SUBMISSION_PATH), exist_ok=True)
submission_df.to_csv(SUBMISSION_PATH, index=False)

print(f"\n--- 競賽提交完成 ---")
print(f"生成的提交檔案已儲存至: {SUBMISSION_PATH}")
print("預測結果範例:\n", submission_df.head())

--- 1. 資料載入與清洗 ---
訓練集貼文數: 47890
情感類別數: 6 (joy, fear, anger, surprise, sadness, disgust)

--- 2. 特徵工程 (TF-IDF) ---
模型輸入維度 (Input Shape): 10000

--- 3. 標籤 One-Hot 編碼 ---

--- 4. 建立並訓練 Keras DNN 模型 ---


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 10000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     1,280,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 6)              │           390 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ softmax (Softmax)               │ (None, 6)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,288,774 (4.92 MB)

 Trainable params: 1,288,774 (4.92 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
674/674 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.5648 - loss: 1.1822 - val_accuracy: 0.6020 - val_loss: 1.0730
Epoch 2/10
674/674 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.6711 - loss: 0.8905 - val_accuracy: 0.5947 - val_loss: 1.1097
Epoch 3/10
674/674 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.7317 - loss: 0.7224 - val_accuracy: 0.5815 - val_loss: 1.2153
Epoch 4/10
674/674 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.7959 - loss: 0.5625 - val_accuracy: 0.5653 - val_loss: 1.4081
Epoch 5/10
674/674 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.8607 - loss: 0.3978 - val_accuracy: 0.5448 - val_loss: 1.6681
Epoch 6/10
674/674 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9143 - loss: 0.2605 - val_accuracy: 0.5387 - val_loss: 2.0030
Epoch 7/10
674/674 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9449 - loss: 0.1707 - val_accuracy: 0.5396 - val_loss: 2.3770
Epoch 8/10
674/674 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.9623 - loss: 0.1180 - val_accuracy: 0.

In [16]:
import pandas as pd
import json
import os
import numpy as np
import re
from time import time
import keras
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
# 導入修正過擬合所需的 Keras 組件
from keras.models import Model
from keras.layers import Input, Dense, ReLU, Softmax, Dropout
from keras.callbacks import EarlyStopping
from keras import regularizers # 用於 L2 正則化 (可選)
from sklearn.metrics import classification_report


# --- 檔案路徑設定 ---
BASE_PATH = "DM kaggle/dm-lab-2-private-competition/" 
SUBMISSION_PATH = "DM kaggle/dm-lab-2-private-competition/submission.csv"

JSON_PATH = BASE_PATH + "final_posts.json"
EMOTION_PATH = BASE_PATH + "emotion.csv"
SPLIT_PATH = BASE_PATH + "data_identification.csv"

# --- 1. 資料載入與清洗 ---
print("--- 1. 資料載入與清洗 ---")
try:
    with open(JSON_PATH, 'r') as f:
        raw_data = json.load(f)
    
    posts_data = [{'id': item['root']['_source']['post']['post_id'],
                   'text': item['root']['_source']['post']['text']} for item in raw_data]
    df_posts = pd.DataFrame(posts_data)

    df_emotions = pd.read_csv(EMOTION_PATH)
    df_split = pd.read_csv(SPLIT_PATH)

    df_combined = df_posts.merge(df_split, on='id', how='left').merge(df_emotions, on='id', how='left')

    train_df = df_combined[df_combined['split'] == 'train'].copy().reset_index(drop=True)
    test_df_raw = df_combined[df_combined['split'] == 'test'].copy().reset_index(drop=True)
    
    EMOTIONS = train_df['emotion'].unique()
    NUM_CLASSES = len(EMOTIONS)
    print(f"訓練集貼文數: {len(train_df)}")

except FileNotFoundError as e:
    print(f"致命錯誤：找不到檔案。請檢查檔案路徑。錯誤: {e}")
    raise

# 文本清洗函數
def clean_text(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'\[NAME\]|\[RELIGION\]', '', text) 
    text = re.sub(r'[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF]+', '', text) 
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text

train_df['cleaned_text'] = train_df['text'].apply(clean_text)
test_df_raw['cleaned_text'] = test_df_raw['text'].apply(clean_text)


# --- 2. 特徵工程 (TF-IDF) ---
print("\n--- 2. 特徵工程 (TF-IDF) ---")
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), stop_words='english')
X_train_sparse = vectorizer.fit_transform(train_df['cleaned_text'])
X_test_sparse = vectorizer.transform(test_df_raw['cleaned_text'])
X_train = X_train_sparse.toarray()
X_test = X_test_sparse.toarray()

input_shape = X_train.shape[1]
print(f"模型輸入維度 (Input Shape): {input_shape}")


# --- 3. 處理類別標籤 (One-Hot Encoding) ---
label_encoder = LabelEncoder()
label_encoder.fit(train_df['emotion'])
y_train_encoded = label_encoder.transform(train_df['emotion'])
y_train_onehot = keras.utils.to_categorical(y_train_encoded, num_classes=NUM_CLASSES)


# --- 4. 建立 DL 模型 (加入 Dropout 和 L2 正則化) ---
print("\n--- 4. 建立並訓練 Keras DNN 模型 (已加入正則化) ---")

model_input = Input(shape=(input_shape, ))
X = model_input

# 1st hidden layer: 加入 L2 正則化
X_W1 = Dense(units=128, kernel_regularizer=regularizers.l2(0.0005))(X)
H1 = ReLU()(X_W1)
H1_Drop = Dropout(0.2)(H1) # 20% Dropout

# 2nd hidden layer: 加入 L2 正則化
H1_W2 = Dense(units=64, kernel_regularizer=regularizers.l2(0.0005))(H1_Drop)
H2 = ReLU()(H1_W2)

# Output layer
H2_W3 = Dense(units=NUM_CLASSES)(H2) 
model_output = Softmax()(H2_W3)

model = Model(inputs=[model_input], outputs=[model_output])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()


# --- 5. 訓練模型 (加入 Early Stopping) ---
epochs = 100 # 設置較高 Epochs，讓 EarlyStopping 決定最佳停止點
batch_size = 64

# Early Stopping: 監控 val_loss, 連續 3 個 Epoch 無改善則停止，並恢復最佳權重
early_stopping = EarlyStopping(monitor='val_loss', 
                               patience=3, 
                               restore_best_weights=True,
                               verbose=1)

start_time = time()
history = model.fit(X_train, y_train_onehot, 
                    epochs=epochs, 
                    batch_size=batch_size, 
                    validation_split=0.1, 
                    callbacks=[early_stopping], # 加入 Early Stopping
                    verbose=1)
end_time = time()
print(f"\n模型訓練完成。耗時: {end_time - start_time:.2f} 秒")


# --- 6. 預測與提交檔案生成 ---

# 訓練集性能檢查 (驗證正則化效果)
pred_probabilities_train = model.predict(X_train, batch_size=256)
pred_indices_train = np.argmax(pred_probabilities_train, axis=1)
pred_emotion_train = label_encoder.inverse_transform(pred_indices_train)
target_names = label_encoder.classes_
print("\n訓練集性能報告 (修正過擬合後):\n", classification_report(train_df['emotion'], pred_emotion_train, target_names=target_names))


# 預測測試集
pred_probabilities_test = model.predict(X_test, batch_size=256)
pred_indices_test = np.argmax(pred_probabilities_test, axis=1)
pred_emotion_test = label_encoder.inverse_transform(pred_indices_test)

# 準備提交格式
submission_df = test_df_raw[['id']].copy()
submission_df['emotion'] = pred_emotion_test
submission_df.columns = ['id', 'emotion']

# 儲存為 CSV 檔案
os.makedirs(os.path.dirname(SUBMISSION_PATH), exist_ok=True)
submission_df.to_csv(SUBMISSION_PATH, index=False)

print(f"\n--- 競賽提交完成 ---")
print(f"生成的提交檔案已儲存至: {SUBMISSION_PATH}")

--- 1. 資料載入與清洗 ---
訓練集貼文數: 47890

--- 2. 特徵工程 (TF-IDF) ---
模型輸入維度 (Input Shape): 10000

--- 4. 建立並訓練 Keras DNN 模型 (已加入正則化) ---


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 10000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │     1,280,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_4 (ReLU)                  │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_5 (ReLU)                  │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 6)              │           390 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ softmax_2 (Softmax)             │ (None, 6)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,288,774 (4.92 MB)

 Trainable params: 1,288,774 (4.92 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
674/674 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.5527 - loss: 1.3029 - val_accuracy: 0.5840 - val_loss: 1.2515
Epoch 2/100
674/674 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.6117 - loss: 1.2073 - val_accuracy: 0.6033 - val_loss: 1.2402
Epoch 3/100
674/674 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.6347 - loss: 1.1725 - val_accuracy: 0.6003 - val_loss: 1.2483
Epoch 4/100
674/674 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.6509 - loss: 1.1472 - val_accuracy: 0.6028 - val_loss: 1.2570
Epoch 5/100
674/674 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.6627 - loss: 1.1229 - val_accuracy: 0.6049 - val_loss: 1.2660
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.

模型訓練完成。耗時: 28.88 秒
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


/Users/lizongtao/Desktop/DM2025Lab2/DM2025-Lab2-Exercise/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/lizongtao/Desktop/DM2025Lab2/DM2025-Lab2-Exercise/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/lizongtao/Desktop/DM2025Lab2/DM2025-Lab2-Exercise/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_divi


訓練集性能報告 (修正過擬合後):
               precision    recall  f1-score   support

       anger       0.60      0.70      0.64     10694
     disgust       0.00      0.00      0.00      1183
        fear       0.74      0.18      0.29      2009
         joy       0.73      0.89      0.80     23797
     sadness       0.60      0.32      0.42      3926
    surprise       0.60      0.36      0.45      6281

    accuracy                           0.68     47890
   macro avg       0.54      0.41      0.43     47890
weighted avg       0.65      0.68      0.65     47890

64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

--- 競賽提交完成 ---
生成的提交檔案已儲存至: DM kaggle/dm-lab-2-private-competition/submission.csv


In [24]:
import pandas as pd
import json
import os
import numpy as np
import re
from time import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
# 引入 LightGBM 模型
import lightgbm as lgb 

# --- 檔案路徑設定 ---
# 這裡使用上一輪自動切換成功的路徑，以確保檔案能被讀取。
BASE_PATH = "DM kaggle/dm-lab-2-private-competition/" 
SUBMISSION_PATH = "DM kaggle/dm-lab-2-private-competition/submission.csv"

JSON_PATH = BASE_PATH + "final_posts.json"
EMOTION_PATH = BASE_PATH + "emotion.csv"
SPLIT_PATH = BASE_PATH + "data_identification.csv"

# --- 1. 資料載入與清洗 ---
print("--- 1. 資料載入與清洗 ---")
try:
    with open(JSON_PATH, 'r') as f:
        raw_data = json.load(f)
    
    posts_data = [{'id': item['root']['_source']['post']['post_id'],
                   'text': item['root']['_source']['post']['text']} for item in raw_data]
    df_posts = pd.DataFrame(posts_data)

    df_emotions = pd.read_csv(EMOTION_PATH)
    df_split = pd.read_csv(SPLIT_PATH)

    df_combined = df_posts.merge(df_split, on='id', how='left').merge(df_emotions, on='id', how='left')

    train_df = df_combined[df_combined['split'] == 'train'].copy().reset_index(drop=True)
    test_df_raw = df_combined[df_combined['split'] == 'test'].copy().reset_index(drop=True)
    
    EMOTIONS = train_df['emotion'].unique()
    NUM_CLASSES = len(EMOTIONS)
    print(f"訓練集貼文數: {len(train_df)}")

except FileNotFoundError as e:
    print(f"致命錯誤：找不到檔案。請檢查檔案路徑。錯誤: {e}")
    raise

# 文本清洗函數 (與前一個方案相同)
def clean_text(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'\[NAME\]|\[RELIGION\]', '', text) 
    text = re.sub(r'[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF]+', '', text) 
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text

train_df['cleaned_text'] = train_df['text'].apply(clean_text)
test_df_raw['cleaned_text'] = test_df_raw['text'].apply(clean_text)


# --- 2. 特徵工程 (Feature Engineering) ---
print("\n--- 2. 特徵工程 (TF-IDF) ---")
# LightGBM 處理稀疏矩陣時性能優異，這裡仍使用 10000 維度的 TF-IDF
vectorizer = TfidfVectorizer(max_features=10000, 
                             ngram_range=(1, 2), 
                             stop_words='english')

# Fit and Transform 訓練集
X_train_sparse = vectorizer.fit_transform(train_df['cleaned_text'])

# Transform 測試集
X_test_sparse = vectorizer.transform(test_df_raw['cleaned_text'])

print(f"模型輸入維度 (Input Shape): {X_train_sparse.shape[1]}")


# --- 3. 處理類別標籤 (Label Encoding) ---
print("\n--- 3. 標籤整數編碼 ---")
# LightGBM 要求整數編碼 (無需 One-Hot)
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(train_df['emotion'])
y_train = y_train_encoded

X_train = X_train_sparse
X_test = X_test_sparse


# --- 4. 訓練 LightGBM 模型 (使用稀疏矩陣輸入) ---
print("\n--- 4. 訓練 LightGBM 模型 ---")

# LightGBM 模型設定 (Gradient Boosting on Decision Trees)
lgb_model = lgb.LGBMClassifier(
    objective='multiclass', 
    num_class=NUM_CLASSES, 
    metric='multi_logloss',
    n_estimators=1000,         # 樹的數量 (可調參數)
    learning_rate=0.05,        # 學習率 (可調參數)
    num_leaves=31,             # 每棵樹的最大葉子數 (可調參數)
    random_state=42,
    n_jobs=-1,                 # 使用所有核心進行加速
    verbose=-1                 # 關閉 LightGBM 訓練訊息
)

start_time = time()
# LightGBM 可以直接使用稀疏矩陣 (X_train) 和整數標籤 (y_train)
lgb_model.fit(X_train, y_train)
end_time = time()
print(f"LightGBM 模型訓練完成。耗時: {end_time - start_time:.2f} 秒")


# --- 5. 預測與提交檔案生成 ---

# 訓練集性能檢查
y_train_pred_encoded = lgb_model.predict(X_train)
target_names = label_encoder.classes_
print("\n訓練集性能報告 (LightGBM):\n", classification_report(y_train, y_train_pred_encoded, target_names=target_names))


# 對測試集進行預測
y_test_pred_encoded = lgb_model.predict(X_test)

# 將類別索引轉回情緒名稱
pred_emotion_test = label_encoder.inverse_transform(y_test_pred_encoded)

# 準備提交格式
submission_df = test_df_raw[['id']].copy()
submission_df['emotion'] = pred_emotion_test
submission_df.columns = ['id', 'emotion']

# 儲存為 CSV 檔案
os.makedirs(os.path.dirname(SUBMISSION_PATH), exist_ok=True)
submission_df.to_csv(SUBMISSION_PATH, index=False)

print(f"\n--- 競賽提交完成 ---")
print(f"生成的提交檔案已儲存至: {SUBMISSION_PATH}")
print("預測結果範例:\n", submission_df.head())

ModuleNotFoundError: No module named 'lightgbm'